# Étape 2 — Édition OCRTransforme `<Livre>_OCR.txt` en `<Livre>_EDIT.txt` : correction des erreurs de reconnaissance, puis passe de raccord entre les blocs.**Le texte de l'auteur n'est jamais réécrit.** Seules les erreurs manifestes d'OCR sont corrigées.---**Ce notebook n'est qu'une interface.** Toute la logique vit dans le paquet`theatre_editor`. On y monte le Drive, on installe les dépendances, on surchargeéventuellement la configuration, puis on lance l'étape.**Cette étape est reprenable.** Si Colab coupe, relancez la celluled'exécution : le travail déjà validé ne sera pas refait, et vous ne repaierezaucun appel.

## 1. Dépendances et montage du Drive

In [ ]:
# Installation des dépendances du pipeline.!pip install -q -U openai pymupdf python-docxfrom google.colab import drivedrive.mount("/content/drive")

## 2. Récupération du codeLe dépôt [`elyeskaak/texte_troupe_theatre`](https://github.com/elyeskaak/texte_troupe_theatre)est **privé** : son clone exige un jeton d'accès personnel.**À faire une fois.**1. GitHub → *Settings* → *Developer settings* → *Personal access tokens* →   *Fine-grained tokens* → **Generate new token**   - *Repository access* : **uniquement** `texte_troupe_theatre`   - *Permissions* → *Repository permissions* → **Contents : Read-only**   Rien de plus. Un jeton limité à la lecture d'un seul dépôt ne peut rien   casser s'il fuite.2. Colab → panneau latéral **🔑 Secrets** → ajouter `GITHUB_TOKEN` →   activer « Accès au notebook ».Si vous préférez ne pas créer de jeton, utilisez l'option B : déposez le dossier`theatre_editor/` directement sur votre Drive.

In [ ]:
# --- Option A : clone du dépôt privé -----------------------------------DEPOT_COMPTE = "elyeskaak"DEPOT_NOM = "texte_troupe_theatre"DOSSIER_PROJET = f"/content/{DEPOT_NOM}"import osimport subprocessimport sysfrom google.colab import userdatatry:    jeton = userdata.get("GITHUB_TOKEN")except Exception:    jeton = Noneif not jeton:    raise RuntimeError(        "Secret GITHUB_TOKEN introuvable.\n"        "Panneau « 🔑 Secrets » → ajouter GITHUB_TOKEN "        "→ activer « Accès au notebook »."    )# L'URL contient le jeton : elle ne doit JAMAIS être affichée, ni figurer dans# un message d'erreur. Les sorties de git sont donc capturées, jamais relayées.url = f"https://{jeton}@github.com/{DEPOT_COMPTE}/{DEPOT_NOM}.git"if os.path.isdir(DOSSIER_PROJET):    commande = ["git", "-C", DOSSIER_PROJET, "pull", "--quiet"]else:    commande = ["git", "clone", "--quiet", url, DOSSIER_PROJET]resultat = subprocess.run(commande, capture_output=True, text=True)if resultat.returncode != 0:    raise RuntimeError(        "Récupération du code impossible.\n"        "Vérifiez que le jeton est valide et qu'il donne accès en lecture "        f"au dépôt {DEPOT_COMPTE}/{DEPOT_NOM}."    )if DOSSIER_PROJET not in sys.path:    sys.path.insert(0, DOSSIER_PROJET)print("Code récupéré :", DOSSIER_PROJET)

In [ ]:
# --- Option B : le dossier theatre_editor/ est sur votre Drive ---------# Décommentez ces lignes et ajustez le chemin, puis n'exécutez PAS l'option A.# import sys# DOSSIER_PROJET = "/content/drive/MyDrive/texte_troupe_theatre"# if DOSSIER_PROJET not in sys.path:#     sys.path.insert(0, DOSSIER_PROJET)

## 3. Configuration`config.py` porte toutes les valeurs par défaut. Les surcharges ci-dessous nevalent que pour cette session : elles ne modifient pas le fichier.**Vérifiez le dossier de travail** avant de continuer.

In [ ]:
from pathlib import Pathfrom theatre_editor import config# Dossier Drive contenant les PDF et recevant toutes les sorties.config.DOSSIER_DRIVE = Path("/content/drive/MyDrive/Troupe 122 - 2026-27")print("Dossier de travail :", config.DOSSIER_DRIVE)print("Existe             :", config.DOSSIER_DRIVE.is_dir())

## 4. Clé API

In [ ]:
# La clé API est lue depuis les Secrets de Colab.##   panneau latéral « 🔑 Secrets » → ajouter OPENAI_API_KEY#   → activer « Accès au notebook »## Ainsi la clé n'apparaît jamais dans le notebook ni dans ses sorties.from theatre_editor.utils import iotry:    io.charger_cle_api()    print("Clé API trouvée.")except RuntimeError as erreur:    print(erreur)

## 5. Vérification des modèles

In [ ]:
# Contrôle que les identifiants de config.py existent bien sur ce compte.# Deux secondes ici évitent de découvrir une faute de frappe après trois# heures de traitement.from theatre_editor.utils import apiapi.verifier_modeles_configures()

## 6. Réglages de l'édition`PAGES_PAR_BLOC` est le réglage le plus sensible. **Ne le changez pas au milieud'un livre** : les blocs déjà édités ne seraient plus alignés, et l'étape 3refuserait de valider.

In [ ]:
config.PAGES_PAR_BLOC = 8          # 6 à 10 est une bonne plageconfig.LIGNES_CONTEXTE_RACCORD = 50config.RATIO_MINIMAL_LONGUEUR = 0.80print("Modèle d'édition :", config.MODEL_EDITION)print("Modèle de raccord:", config.MODEL_RACCORD)print("Pages par bloc   :", config.PAGES_PAR_BLOC)

## 7. Aperçu du découpageMontre en combien de blocs chaque livre sera découpé, et ce qui est déjà fait.

In [ ]:
from theatre_editor.utils import blocks, iofor chemin in io.lister_fichiers_ocr(config.DOSSIER_DRIVE):    nom = io.nom_livre_depuis_ocr(chemin)    chemins = io.resoudre_chemins(nom, chemin.parent)    pages = blocks.decouper_en_pages(io.lire_texte(chemin))    liste = blocks.former_blocs(pages, config.PAGES_PAR_BLOC)    faits = sum(        1 for b in liste if io.unite_terminee(chemins.bloc_json(b.numero))    )    raccords = sum(        1        for numero in range(1, max(1, len(liste)))        if io.unite_terminee(chemins.raccord_json(numero))    )    print(f"{nom}")    print(f"   {len(pages)} pages → {len(liste)} blocs")    print(f"   {faits}/{len(liste)} bloc(s) édité(s), "          f"{raccords}/{max(0, len(liste) - 1)} raccord(s) fait(s)")

## 8. LancementLes deux passes s'enchaînent : édition des blocs, puis raccord des jonctions.Reprenable à l'unité près.

In [ ]:
from theatre_editor import editionresultats = edition.executer(config.DOSSIER_DRIVE)

## 9. Contrôle du résultatAffiche le début de chaque `EDIT.txt` et la structure que l'étape 4 y verra.C'est le moment de vérifier que la convention typographique est bien appliquée.

In [ ]:
for resultat in resultats:    chemins = io.resoudre_chemins(resultat.nom, config.DOSSIER_DRIVE)    print("=" * 72)    print(resultat.nom, "—", resultat.statut)    print("=" * 72)    if not chemins.edit.exists():        print("Aucun fichier édité.")        continue    texte = io.lire_texte(chemins.edit)    print(texte[:1200])    print()    print(blocks.rapport_classification(blocks.construire_index_structure(texte)))

## 10. Journal

In [ ]:
# Journal détaillé de l'étape : un enregistrement par appel API, avec sa# date, son modèle, son response_id, sa durée et sa consommation de jetons.import jsonchemin = config.DOSSIER_DRIVE / config.NOM_JOURNAL.format(etape="edition")if chemin.exists():    journal = json.loads(chemin.read_text(encoding="utf-8"))    print("Dernière exécution :", journal["derniere_execution"])    print("Configuration      :", json.dumps(journal["configuration"], ensure_ascii=False))    print()    for nom, bilan in journal["livres"].items():        print(f"{nom} : {json.dumps(bilan, ensure_ascii=False)}")    jetons = sum(        (appel.get("tokens_entree") or 0) + (appel.get("tokens_sortie") or 0)        for appel in journal["appels"]    )    print()    print(f"{len(journal['appels'])} appel(s) journalisé(s), {jetons:,} jetons".replace(",", " "))else:    print("Aucun journal : l'étape n'a pas encore été lancée.")